In [1]:
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime

# today
today = datetime.today().strftime("%y%m%d")

## Pull patents for manual review

In [2]:
# Connect to database
db = duckdb.connect("patents.db")
db.execute("PRAGMA memory_limit='3GB'")
db.execute("PRAGMA temp_directory='C:/temp/duckdb_spill'")
db.execute("PRAGMA threads=2")

In [3]:
# Show tables in database
db.sql("SHOW TABLES")

┌─────────────────────────┐
│          name           │
│         varchar         │
├─────────────────────────┤
│ patents_classified      │
│ patents_embeddings      │
│ run_260710_1614         │
│ run_260710_1614_reverse │
│ run_260713_0858         │
│ run_260713_0858_reverse │
└─────────────────────────┘

#### Get data from patents_classified

In [11]:
# Create dataframe from classified patents
data = db.sql("SELECT * FROM patents_classified").df()

In [ ]:
# If this is a review on an updated dataset, meaning the columns scope_curated and pillar_curated already exist, perform this filtering step
#data = data[data['scope_curated'].isna()]

In [ ]:
data['pred_combined'].value_counts()

scope_curated
out    18242
in     12403
Name: count, dtype: int64

In [14]:
print(f"Of {len(data)} patents, {round(len(data[data["pred_combined"] == "out"])/len(data), 2)*100}% were removed during the ML classification.") 
print(f"\nOf the remaining {len(data[data["pred_combined"] == "in"])} patents, {round(len(data[data["scope_LLM"] == "in"])/len(data[data["pred_combined"] == "in"]), 2)*100}% ({len(data[data["scope_LLM"] == "in"])}) were classified as in scope by the LLM")

Of 31370 patents, 23.0% were removed during the ML classification.

Of the remaining 24046 patents, 43.0% (10341) were classified as in scope by the LLM


#### Create filter mask for curated scope and pillar 

In [15]:
# Scenario 3
has_pillar = (data['pillar_LLM'] != "NA") | (data['pred_pillar'] != "NA")
conf       = data['confidence_LLM']
proba      = data['proba_scope']

conditions = [
    (data['scope_LLM'] == "NA") & (data['pred_combined'] == 'in'),
    (data['scope_LLM'] == 'in') & (conf >= 5),
    (conf <= 4) & (proba > 0.8) & (data['pred_pillar'] != "NA"),
    (data['scope_LLM'] == 'out') & (conf == 2) & (proba < 0.6) & (data['pred_pillar'] == "NA"),
    (data['scope_LLM'] == 'out') & (conf == 1) & (proba < 0.8),
]
choices = ['manual_review', 'in', 'in', 'out', 'out']

#### Assign curated scope/pillar or manual review

In [16]:
# Create curated scope and pillar columns with decision criteria
data['scope_curated'] = np.select(conditions, choices, default='manual_review')
data['scope_curated'] = np.where(data['pred_combined'] == 'out', 'out', data['scope_curated'])
data['pillar_curated'] = np.where(data['scope_curated'] == 'out', np.nan, data['pillar_LLM'].replace('NA', np.nan).fillna(data['pred_pillar'].replace('NA', np.nan)))
data['pillar_curated'] = np.where(data['pred_combined'] == 'out', np.nan, data['pillar_curated'])
data['date_review'] = today

In [17]:
# How many patents have to go through manual review?
data['scope_curated'].value_counts()

scope_curated
out              16492
in               11968
manual_review     2910
Name: count, dtype: int64

In [ ]:
len(data)

31370

In [21]:
data[data['scope_curated'] == "manual_review"]['confidence_LLM'].value_counts()

confidence_LLM
2.0    2390
3.0     358
4.0      81
6.0      41
1.0      16
7.0       8
5.0       2
Name: count, dtype: int64

In [28]:
data[data['scope_curated'] == "manual_review"].loc[:,"proba_scope":"pillar_LLM"]

,proba_scope,pred_combined,pred_pillar,date_ML,scope_LLM,confidence_LLM,pillar_LLM
15,0.781788,in,PB,260710,out,2.0,NA
87,0.634232,in,NA,260710,out,2.0,NA
233,0.431873,in,CC,260710,out,2.0,NA
284,0.457384,in,PB,260710,out,2.0,NA
297,0.474212,in,PB,260710,out,2.0,NA
...,...,...,...,...,...,...,...
10728,0.742741,in,PB,260710,out,2.0,NA
10735,0.791046,in,NA,260710,out,2.0,NA
10751,0.608197,in,NA,260710,out,3.0,NA
10753,0.701477,in,CM,260710,out,2.0,NA


#### Save borderline patents for manual review

In [11]:
# Export patents for review
review = data[data['scope_curated'] == 'manual_review']
review.to_csv(f"data/patents_for_review_{today}.csv")

In [ ]:
# review = pd.read_csv(f"data/patents_for_review_{today}.csv")

#### Save (automatically) curated scope and pillar back to patents_classified

In [24]:
# filter for the patents with clear scope and pillar assigment
data = data[(data['scope_curated'] != 'manual_review') & (~data['scope_curated'].isna())]

In [31]:
# create columns in patents_classified if they don't already exist
db.sql(f"ALTER TABLE patents_classified ADD COLUMN IF NOT EXISTS scope_curated VARCHAR")
db.sql(f"ALTER TABLE patents_classified ADD COLUMN IF NOT EXISTS pillar_curated VARCHAR")

In [25]:
# add to database
db.register('data', data[['id', 'scope_curated', 'pillar_curated']])
db.sql(f"""
    UPDATE patents_classified
    SET scope_curated     = data.scope_curated,
        pillar_curated    = data.pillar_curated
    FROM data
    WHERE patents_classified.id = data.id
""")

In [3]:
db.close()

## Assign manually curated scope and pillar to respective patents

In [41]:
# Connect to database
db = duckdb.connect("patents.db")

In [9]:
# load manually reviewed data
reviewed_data = pd.read_csv(f"data/patents_reviewed_260716.csv")

#### Add scope and pillar back to original dataset, using publication ID

In [6]:
# create columns in patents_classified if they don't already exist
db.sql(f"ALTER TABLE patents_classified ADD COLUMN IF NOT EXISTS date_review VARCHAR")
db.sql(f"ALTER TABLE patents_classified ADD COLUMN IF NOT EXISTS reviewer VARCHAR")

In [10]:
# add to database
db.register('reviewed_data', reviewed_data[['id', 'scope_curated', 'pillar_curated', 'date_review', 'reviewer']])
db.sql(f"""
    UPDATE patents_classified
    SET scope_curated     = reviewed_data.scope_curated,
        pillar_curated    = reviewed_data.pillar_curated,
        date_review       = reviewed_data.date_review,
        reviewer          = reviewed_data.reviewer
    FROM reviewed_data
    WHERE patents_classified.id = reviewed_data.id
""")

In [42]:
data = db.sql("SELECT * FROM patents_classified").df()

In [12]:
data[(data['scope_curated'] == "in") & (~data['pillar_curated'].isin(["PB", "F", "CM", "CC"]))]

,id,title,abstract,application_number,assignee_cities,assignee_countries,assignee_names,cpc,family_count,family_id,...,pillar_LLM,plant_based_LLM,fermentation_LLM,cultivated_LLM,cross_cutting_LLM,status_LLM,stop_reason_LLM,date_LLM,date_review,reviewer


In [44]:
db.close()